# Fly Tracker

In this lab, you'll use a Python script to track the position and velocity of a fruit fly from video recordings. The script will detect the fly against a light background, track its movement frame-by-frame, and output position and velocity data as CSV files, along with plots.

**At the end of this notebook, you'll be able to:**
- Run the fly tracker script on your own video recordings
- Load and visualize tracking and velocity data in Python
- Customize your plots for your lab report

<hr>

## Part 1: Running the Fly Tracker Script

The fly tracker is a Python script that you'll run from the command line. Follow the steps below to get set up.

### Step 1: Open Anaconda Prompt

1. Click the **Windows search bar** (bottom-left of your screen) and type **Anaconda**.
2. Click **Anaconda Prompt** to open it. You should see a black terminal window.

:::{note}
This is intended to be run on the BIPN 145 Windows lab computers in York 1310. If you are running this on your own Mac computer, you can search for Python and run it that way. If you're running it on your own Windows computer, you'll need to download Python or Anaconda. 
:::

### Step 2: Download the Script

In the Anaconda Prompt, first use the following to move into *your* folder. Replace USER with your user name.

:::{tip}
**Copying and pasting into the Anaconda Prompt:** You can copy commands from this page by clicking the copy button (📋) at the top-right of each code block and using Ctrl+V to paste into the command line.
:::

```
cd C:\Users\USER
```

 Then, run the following command to download the fly tracker script:

```
curl -o BIPN145_flytrack.py https://raw.githubusercontent.com/ajuavinett/fly_tracker/master/BIPN145_flytrack.py
```

This will save the file `BIPN145_flytrack.py` to whatever folder your Anaconda Prompt is currently in (usually `C:\Users\YourName`). If you already have an older version of the script, this will overwrite it with the latest version. You can also download the script manually from the [fly_tracker GitHub page](https://github.com/ajuavinett/fly_tracker).

### Step 3: Know Where Your Video File(s) Are

Make sure you know where your fly video file(s) (`.avi`, `.mp4`, etc.) are saved on your computer. When you run the script, a file picker window will pop up and you can navigate to them from there.

### Step 4: Run the Script

In the Anaconda Prompt, type:

```
python BIPN145_flytrack.py
```
:::{note}
The first time you run the script, it will automatically install any missing Python packages (`numpy`, `opencv-python`, `matplotlib`). This only happens once.
:::

The script will walk you through everything interactively:

1. **Select your video(s)** — A file picker dialog will pop up. Navigate to your video file(s), select them, and click Open.

2. **Set the frame rate** — The script will ask you for the frame rate of your video. Press **Enter** to accept the default (shown in brackets), or type a new value:
   ```
   Frame rate (fps) [15]:
   ```

3. **Draw the ROI** — A window will pop up showing the first frame of your video. **Click and drag** to draw a rectangle around the dish/arena, then press **Enter** or **Space** to confirm. Press **C** to cancel and redraw.

4. **Wait for tracking** — The script will process each frame. You'll see progress updates (10%, 20%, etc.). This will take a few minutes to run per video.

5. **View results** — When done, plots will appear and CSV files will be saved automatically. The plot windows will stay open so you can inspect them and **save them by clicking on the disk icon**.

### Step 5: Find Your Output Files

After the script finishes, you'll find two CSV files for each video in the same folder as your video:

| File | Contents |
| --- | --- |
| `yourvideo_tracking.csv` | Time (s), X position (cm), Y position (cm) |
| `yourvideo_velocity.csv` | Time (s), Velocity (mm/s) |

If you need to edit them, you can open these in Excel, Google Sheets, or load them into Python (see below).

<hr>

## Part 2: Working with Your Data in Python (Optional)

The cells below will help you load your CSV output files and create customized plots in case you need to adjust anything after running the tracking. This is useful if you want to adjust axis labels, colors, or combine data from multiple flies for your lab report.

> **Task**: Run the cell below to import the packages we'll need.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

print('Packages imported!')

Packages imported!


### Upload your CSV files

If you're running this notebook in **Google Colab**, run the cell below to upload your tracking and velocity CSV files. If you're running locally, you can skip this cell and just set the file paths directly.

> **Task**: Run the cell below and upload your `_tracking.csv` and `_velocity.csv` files.

In [2]:
try:
    from google.colab import files
    uploaded = files.upload()
    filenames = list(uploaded.keys())
    tracking_file = [f for f in filenames if 'tracking' in f][0]
    velocity_file = [f for f in filenames if 'velocity' in f][0]
    print(f'Tracking file: {tracking_file}')
    print(f'Velocity file: {velocity_file}')
except ImportError:
    # Running locally — set your file paths here
    tracking_file = 'yourvideo_tracking.csv'  # <-- change this
    velocity_file = 'yourvideo_velocity.csv'  # <-- change this
    print(f'Using local files: {tracking_file}, {velocity_file}')

Using local files: yourvideo_tracking.csv, yourvideo_velocity.csv


### Load the data

Now let's load both CSV files using `numpy`. Each file has a header row, so we skip it with `skiprows=1`.

> **Task**: Run the cell below to load your data.

In [3]:
# Load tracking data: columns are Time (s), X (cm), Y (cm)
tracking = np.loadtxt(tracking_file, delimiter=',', skiprows=1)
time_pos = tracking[:, 0]
x_pos = tracking[:, 1]
y_pos = tracking[:, 2]

# Load velocity data: columns are Time (s), Velocity (mm/s)
vel_data = np.loadtxt(velocity_file, delimiter=',', skiprows=1)
time_vel = vel_data[:, 0]
velocity = vel_data[:, 1]

print(f'Tracking data: {len(time_pos)} frames')
print(f'Velocity data: {len(time_vel)} time bins')
print(f'Mean velocity: {np.nanmean(velocity):.2f} mm/s')
print(f'SD velocity:   {np.nanstd(velocity):.2f} mm/s')

FileNotFoundError: yourvideo_tracking.csv not found.

### Plot the fly path

The plot below shows the fly's path through the arena, color-coded by time. Earlier positions are shown in purple/blue, and later positions in yellow/green.

> **Task**: Run the cell below to plot the fly path. Edit the axis labels, title, or colormap (`cmap`) if you'd like to customize it.

In [ ]:
# Set the diameter of your dish (should match what you used in the script)
diameter = 4  # cm

fig, ax = plt.subplots(figsize=(6, 6))

# Create color-coded path
valid = ~np.isnan(x_pos) & ~np.isnan(y_pos)
points = np.array([x_pos[valid], y_pos[valid]]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)

lc = LineCollection(segments, cmap='viridis', linewidth=2)
lc.set_array(time_pos[valid][:-1])
ax.add_collection(lc)

ax.set_xlim(0, diameter)
ax.set_ylim(diameter, 0)  # Inverted to match video orientation
ax.set_aspect('equal')

# --- Customize these labels for your report ---
ax.set_xlabel('X-coordinate (cm)', fontsize=12)
ax.set_ylabel('Y-coordinate (cm)', fontsize=12)
ax.set_title('Fly Path', fontsize=14)

cbar = fig.colorbar(lc, ax=ax, orientation='horizontal', pad=0.1)
cbar.set_label('Time (s)')

plt.tight_layout()
plt.show()

### Plot velocity over time

The plot below shows how the fly's velocity changes over the course of the recording.

> **Task**: Run the cell below to plot the velocity trace. Edit the labels, colors, or axis limits as needed.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(time_vel, velocity, linewidth=1.5, color='steelblue')

# --- Customize these for your report ---
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Velocity (mm/s)', fontsize=12)
ax.set_title('Fly Velocity Over Time', fontsize=14)

# Uncomment the line below to set custom axis limits:
# ax.set_xlim(0, 60)   # e.g., first 60 seconds
# ax.set_ylim(0, 20)   # e.g., max 20 mm/s

plt.tight_layout()
plt.show()

print(f'Mean velocity: {np.nanmean(velocity):.2f} mm/s')
print(f'SD velocity:   {np.nanstd(velocity):.2f} mm/s')

### Compare multiple flies *(optional)*

If you have velocity data from multiple flies, you can plot them together and compute summary statistics. 

> **Task**: Upload additional velocity CSV files and add their filenames to the list below.

In [ ]:
# Add your velocity CSV filenames here
velocity_files = [
    'fly1_velocity.csv',   # <-- change these
    'fly2_velocity.csv',
    # 'fly3_velocity.csv',
]

fig, ax = plt.subplots(figsize=(10, 5))
all_means = []

for i, vf in enumerate(velocity_files):
    data = np.loadtxt(vf, delimiter=',', skiprows=1)
    t = data[:, 0]
    v = data[:, 1]
    ax.plot(t, v, linewidth=1.5, label=f'Fly {i + 1}')
    all_means.append(np.nanmean(v))

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Velocity (mm/s)', fontsize=12)
ax.set_title('Velocity Comparison', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nMean velocity across flies: {np.mean(all_means):.2f} mm/s')
print(f'SD across flies: {np.std(all_means):.2f} mm/s')

<hr>

## About this Notebook
This notebook was created by [Ashley Juavinett](https://github.com/ajuavinett) for classes at UC San Diego. The fly tracker script is based on MATLAB code by Jeff Stafford.